# 📊 Explorador de Dados CSV

Função para explorar arquivos CSV em um diretório, detectar delimitadores automaticamente e salvar informações detalhadas.

**Índice de Seções:**
1. Importações e Funções de Base
2. Exploração e Relatório em CSV
3. Exploração Detalhada
4. Análise de Arquivo Específico
5. Geração de Documentação em TXT

## Passo 1: Importações e Funções de Base

In [1]:
# IMPORTAÇÕES
import os
import json
import pandas as pd
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════════════
# FUNÇÃO UNIFICADA DE EXPLORAÇÃO DE DADOS COM MÁXIMA INFORMAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════

def detectar_delimitador(caminho, linhas_teste=3):
    """Detecta o delimitador de um arquivo CSV"""
    delimitadores = [';', ',', '\t', '|']
    try:
        with open(caminho, 'r', encoding='utf-8', errors='ignore') as f:
            amostra = ''.join([f.readline() for _ in range(linhas_teste)])
        contagem = {delim: amostra.count(delim) for delim in delimitadores}
        delim_detectado = max(contagem, key=contagem.get)
        return delim_detectado if contagem[delim_detectado] > 0 else ','
    except:
        return ','

def analisar_coluna_detalhada(df, coluna):
    """
    Análise COMPLETA de uma coluna com tipos e intervalos
    Retorna dicionário com até 30 métricas diferentes
    """
    dtype = df[coluna].dtype
    nulos = df[coluna].isnull().sum()
    total_linhas = len(df)
    preenchimento = (1 - nulos / total_linhas) * 100 if total_linhas > 0 else 0
    
    analise = {
        'coluna': coluna,
        'tipo_python': str(dtype),
        'categoria_tipo': 'NUMÉRICO' if pd.api.types.is_numeric_dtype(df[coluna]) else 'CATEGÓRICO',
        'valores_totais': total_linhas,
        'valores_validos': total_linhas - nulos,
        'valores_nulos': nulos,
        'percentual_preenchimento': round(preenchimento, 2),
        'valores_unicos': df[coluna].nunique(),
        'taxa_unica': round((df[coluna].nunique() / (total_linhas - nulos) * 100) if (total_linhas - nulos) > 0 else 0, 2)
    }
    
    # ═══ ANÁLISES PARA TIPOS NUMÉRICOS ═══
    if pd.api.types.is_numeric_dtype(df[coluna]):
        valores = df[coluna].dropna()
        if len(valores) > 0:
            media = valores.mean()
            mediana = valores.median()
            desvio = valores.std()
            variancia = valores.var()
            minimo = valores.min()
            maximo = valores.max()
            amplitude = maximo - minimo
            q1 = valores.quantile(0.25)
            q3 = valores.quantile(0.75)
            iqr = q3 - q1
            
            analise.update({
                'minimo': round(minimo, 6),
                'maximo': round(maximo, 6),
                'amplitude': round(amplitude, 6),
                'intervalo_textual': f"[{round(minimo, 4)}, {round(maximo, 4)}]",
                'media': round(media, 6),
                'mediana': round(mediana, 6),
                'moda': valores.mode()[0] if len(valores.mode()) > 0 else None,
                'desvio_padrao': round(desvio, 6),
                'variancia': round(variancia, 6),
                'coeficiente_variacao_pct': round((desvio / media * 100) if media != 0 else 0, 4),
                'assimetria': round(valores.skew(), 4),
                'curtose': round(valores.kurtosis(), 4),
                'quartil_25': round(q1, 6),
                'quartil_50': round(mediana, 6),
                'quartil_75': round(q3, 6),
                'amplitude_interquartil': round(iqr, 6),
                'limite_inferior_iqr': round(q1 - 1.5 * iqr, 6),
                'limite_superior_iqr': round(q3 + 1.5 * iqr, 6),
                'range_iqr': f"[{round(q1, 4)}, {round(q3, 4)}]",
                'soma': round(valores.sum(), 6),
                'produto': None,  # Evitar overflow
                'percentil_5': round(valores.quantile(0.05), 6),
                'percentil_95': round(valores.quantile(0.95), 6)
            })
    
    # ═══ ANÁLISES PARA TIPOS CATEGÓRICOS ═══
    else:
        valores = df[coluna].dropna()
        if len(valores) > 0:
            comprimentos = valores.astype(str).apply(len)
            top_valores = valores.value_counts().head(5)
            
            # Converter np.int64 para int para JSON serializable
            top_valores_dict = {k: int(v) for k, v in dict(top_valores).items()}
            
            analise.update({
                'valores_top_5': top_valores_dict,
                'frequencia_top': int(top_valores.values[0]) if len(top_valores) > 0 else 0,
                'comprimento_min': comprimentos.min(),
                'comprimento_max': comprimentos.max(),
                'comprimento_medio': round(comprimentos.mean(), 2),
                'comprimento_desvio': round(comprimentos.std(), 2),
                'diversidade': round(len(valores.unique()) / len(valores), 4),
                'valor_mais_frequente': valores.value_counts().idxmax() if len(valores) > 0 else None,
                'frequencia_maxima': int(valores.value_counts().max()) if len(valores) > 0 else 0
            })
    
    return analise

def explorador_csv_completo(pasta=r'D:\OneDrive\Pessoais\Doutorado\Cefet\data', salvar_relatorios=True, gerar_txt=True):
    """
    EXPLORADOR UNIFICADO E PODEROSO DE DADOS CSV
    
    ✅ Remove redundâncias de funções antigas
    ✅ Consolida toda informação em uma única execução
    ✅ Gera relatórios INDIVIDUAIS para CADA arquivo (CSV + TXT)
    ✅ Análise completa com 30+ métricas por coluna
    
    Parâmetros:
        pasta: Diretório com arquivos CSV
        salvar_relatorios: Salva em CSV os dados analisados de cada arquivo
        gerar_txt: Gera arquivo TXT com explicação completa para cada arquivo
    
    SAÍDAS:
        Para CADA arquivo CSV lido, gera:
        - [nome_arquivo]_analise.csv (métricas de todas as colunas)
        - [nome_arquivo]_relatorio.txt (relatório estruturado)
    """
    
    # Validações
    if not os.path.exists(pasta):
        print(f"❌ Pasta não encontrada: {pasta}")
        return
    
    csvs = sorted([f for f in os.listdir(pasta) if f.endswith('.csv')])
    
    if not csvs:
        print("❌ Nenhum arquivo CSV encontrado")
        return
    
    print(f"\n{'='*120}")
    print(f"🚀 EXPLORADOR AVANÇADO DE DADOS CSV - ANÁLISE INDIVIDUAL POR ARQUIVO")
    print(f"{'='*120}")
    print(f"📁 Pasta: {pasta}")
    print(f"📊 Arquivos encontrados: {len(csvs)}\n")
    
    dados_completos = []  # Para retorno da função
    pasta_saida = r'resultados\informacoes_dados'
    os.makedirs(pasta_saida, exist_ok=True)
    
    for i, csv in enumerate(csvs, 1):
        caminho = os.path.join(pasta, csv)
        nome_base = os.path.splitext(csv)[0]  # Nome sem extensão
        
        try:
            # ═══ LEITURA E METADADOS ═══
            delimitador = detectar_delimitador(caminho)
            df = pd.read_csv(caminho, delimiter=delimitador)
            tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
            
            with open(caminho, 'r', encoding='utf-8', errors='ignore') as f:
                num_linhas = sum(1 for _ in f) - 1
            
            num_colunas = len(df.columns)
            taxa_memoria = (tamanho_mb / num_linhas) if num_linhas > 0 else 0
            
            # ═══ CABEÇALHO DO ARQUIVO ═══
            print(f"\n{i}. 📄 {csv}")
            print(f"   {'─'*116}")
            print(f"   📏 Dimensões: {num_linhas:>12,} linhas × {num_colunas:>3} colunas | " 
                  f"💾 Tamanho: {tamanho_mb:>10.2f} MB | " 
                  f"⚖️  {taxa_memoria:>8.4f} MB/linha | "
                  f"🔑 Delimitador: '{delimitador}'")
            print(f"   {'─'*116}\n")
            
            # ═══ DADOS PARA ARQUIVO INDIVIDUAL ═══
            dados_arquivo = []  # Métricas apenas deste arquivo
            relatorio_txt_linhas = []  # Relatório apenas deste arquivo
            
            # Adicionar cabeçalho ao relatório
            relatorio_txt_linhas.append(f"{'═'*100}")
            relatorio_txt_linhas.append(f"ARQUIVO: {csv}")
            relatorio_txt_linhas.append(f"{'═'*100}")
            relatorio_txt_linhas.append(f"Dimensões: {num_linhas:,} linhas × {num_colunas} colunas")
            relatorio_txt_linhas.append(f"Tamanho: {tamanho_mb:.2f} MB | Delimitador: '{delimitador}'")
            
            # ═══ ANÁLISE DE CADA COLUNA ═══
            print(f"   📊 ANÁLISE DETALHADA DE COLUNAS ({num_colunas} total):")
            print(f"   {'─'*116}")
            
            relatorio_txt_linhas.append(f"\n{'-'*100}")
            relatorio_txt_linhas.append("COLUNAS ANALISADAS:")
            relatorio_txt_linhas.append(f"{'-'*100}\n")
            
            for j, col in enumerate(df.columns, 1):
                analise = analisar_coluna_detalhada(df, col)
                dados_arquivo.append(analise)
                dados_completos.append(analise)
                
                # ═══ EXIBIÇÃO FORMATADA ═══
                print(f"\n   [{j:2d}] 📌 {col}")
                print(f"       ├─ Tipo: {analise['tipo_python']:12} ({analise['categoria_tipo']})")
                print(f"       ├─ Preenchimento: {analise['percentual_preenchimento']:6.2f}% " 
                      f"({analise['valores_validos']:,}/{analise['valores_totais']:,})")
                print(f"       ├─ Únicos: {analise['valores_unicos']:>6} ({analise['taxa_unica']:6.2f}%)")
                
                # Informações específicas por tipo
                if analise['categoria_tipo'] == 'NUMÉRICO':
                    print(f"       ├─ Intervalo: {analise['intervalo_textual']}")
                    print(f"       ├─ Centro: Média={analise['media']:>12.4f} | Mediana={analise['mediana']:>12.4f}")
                    print(f"       ├─ Dispersão: σ={analise['desvio_padrao']:>10.4f} | σ²={analise['variancia']:>12.4f} | CV={analise['coeficiente_variacao_pct']:>7.2f}%")
                    print(f"       ├─ Quartis: Q1={analise['quartil_25']:>10.4f} | Q2={analise['quartil_50']:>10.4f} | Q3={analise['quartil_75']:>10.4f} | IQR={analise['amplitude_interquartil']:>10.4f}")
                    print(f"       ├─ Forma: Assimetria={analise['assimetria']:>8.4f} | Curtose={analise['curtose']:>8.4f}")
                    print(f"       └─ Outliers: [{analise['limite_inferior_iqr']:>10.4f}, {analise['limite_superior_iqr']:>10.4f}]")
                    
                    relatorio_txt_linhas.append(f"\n   Coluna: {col} (NUMÉRICO)")
                    relatorio_txt_linhas.append(f"   {'─'*96}")
                    relatorio_txt_linhas.append(f"   Preenchimento: {analise['percentual_preenchimento']:.2f}% | "
                                       f"Valores válidos: {analise['valores_validos']:,} | "
                                       f"Nulos: {analise['valores_nulos']:,}")
                    relatorio_txt_linhas.append(f"   Intervalo: [{analise['minimo']:.6g}, {analise['maximo']:.6g}] "
                                       f"(Amplitude: {analise['amplitude']:.6g})")
                    relatorio_txt_linhas.append(f"   Centro: Média={analise['media']:.6g} | Mediana={analise['mediana']:.6g}")
                    relatorio_txt_linhas.append(f"   Dispersão: Desvio Padrão={analise['desvio_padrao']:.6g} | "
                                       f"Variância={analise['variancia']:.6g}")
                    relatorio_txt_linhas.append(f"   Variação: CV={analise['coeficiente_variacao_pct']:.4f}%")
                    relatorio_txt_linhas.append(f"   Quartis: Q1={analise['quartil_25']:.6g} | "
                                       f"Q3={analise['quartil_75']:.6g} | IQR={analise['amplitude_interquartil']:.6g}")
                    relatorio_txt_linhas.append(f"   Forma: Assimetria={analise['assimetria']:.4f} | "
                                       f"Curtose={analise['curtose']:.4f}")
                    relatorio_txt_linhas.append(f"   Limites Outliers: [{analise['limite_inferior_iqr']:.6g}, "
                                       f"{analise['limite_superior_iqr']:.6g}]")
                
                else:  # CATEGÓRICO
                    print(f"       ├─ Top valor: '{analise['valor_mais_frequente']}' ({analise['frequencia_maxima']} ocorrências)")
                    print(f"       ├─ Comprimento: min={analise['comprimento_min']} | "
                          f"média={analise['comprimento_medio']:.1f} | max={analise['comprimento_max']}")
                    print(f"       └─ Top 5: {analise['valores_top_5']}")
                    
                    relatorio_txt_linhas.append(f"\n   Coluna: {col} (CATEGÓRICO)")
                    relatorio_txt_linhas.append(f"   {'─'*96}")
                    relatorio_txt_linhas.append(f"   Preenchimento: {analise['percentual_preenchimento']:.2f}% | "
                                       f"Valores únicos: {analise['valores_unicos']}")
                    relatorio_txt_linhas.append(f"   Valor mais frequente: '{analise['valor_mais_frequente']}' "
                                       f"({analise['frequencia_maxima']} ocorrências)")
                    relatorio_txt_linhas.append(f"   Comprimento: mín={analise['comprimento_min']} | "
                                       f"médio={analise['comprimento_medio']:.1f} | máx={analise['comprimento_max']}")
                    relatorio_txt_linhas.append(f"   Diversidade: {analise['diversidade']:.4f}")
                    relatorio_txt_linhas.append(f"   Top 5 valores: {analise['valores_top_5']}")
            
            # ═══ SALVAMENTO DE RELATÓRIOS INDIVIDUAIS ═══
            
            # CSV individual
            if salvar_relatorios and dados_arquivo:
                df_metricas = pd.DataFrame(dados_arquivo)
                arquivo_csv = os.path.join(pasta_saida, f'{nome_base}_analise.csv')
                df_metricas.to_csv(arquivo_csv, index=False, encoding='utf-8-sig')
                print(f"\n   ✅ CSV salvo: {arquivo_csv}")
            
            # TXT individual
            if gerar_txt:
                conteudo_txt = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              RELATÓRIO DE EXPLORAÇÃO DE DADOS - {csv[:50]:<30}║
║                                                                              ║
║  Data de Geração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}                                 ║
╚══════════════════════════════════════════════════════════════════════════════╝

{chr(10).join(relatorio_txt_linhas)}

╔══════════════════════════════════════════════════════════════════════════════╗
║                            RESUMO DO ARQUIVO                                ║
╚══════════════════════════════════════════════════════════════════════════════╝

Nome do Arquivo: {csv}
Dimensões: {num_linhas:,} linhas × {num_colunas} colunas
Tamanho: {tamanho_mb:.2f} MB
Total de Colunas Analisadas: {len(dados_arquivo)}
Colunas Numéricas: {sum(1 for x in dados_arquivo if x['categoria_tipo'] == 'NUMÉRICO')}
Colunas Categóricas: {sum(1 for x in dados_arquivo if x['categoria_tipo'] == 'CATEGÓRICO')}

═══════════════════════════════════════════════════════════════════════════════
Fim do Relatório - {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}
═══════════════════════════════════════════════════════════════════════════════
"""
                arquivo_txt = os.path.join(pasta_saida, f'{nome_base}_relatorio.txt')
                with open(arquivo_txt, 'w', encoding='utf-8') as f:
                    f.write(conteudo_txt)
                print(f"   ✅ TXT salvo: {arquivo_txt}")
            
            print(f"\n   ✅ Arquivo processado com sucesso!\n")
            
        except Exception as e:
            print(f"\n   ❌ Erro ao processar {csv}: {str(e)}\n")
    
    print(f"\n{'='*120}")
    print(f"✨ EXPLORAÇÃO CONCLUÍDA COM SUCESSO!")
    print(f"{'='*120}\n")
    print(f"📁 Arquivos salvos em: {pasta_saida}")
    print(f"   Padrão: [nome_arquivo]_analise.csv e [nome_arquivo]_relatorio.txt\n")
    
    return dados_completos

print("✅ EXPLORADOR AVANÇADO carregado com sucesso!")

✅ EXPLORADOR AVANÇADO carregado com sucesso!


## Passo 2: Executar Exploração Completa e Gerar Relatórios

In [2]:
# EXECUTAR O EXPLORADOR AVANÇADO UNIFICADO
print("\n🚀 Iniciando exploração completa de dados...\n")
dados_analise = explorador_csv_completo(salvar_relatorios=True, gerar_txt=True)  # noqa: F405
print("\n✅ Exploração finalizada! Verifique os arquivos em 'relatorio/resultados/informacoes_dados/'")


🚀 Iniciando exploração completa de dados...


🚀 EXPLORADOR AVANÇADO DE DADOS CSV - ANÁLISE INDIVIDUAL POR ARQUIVO
📁 Pasta: D:\OneDrive\Pessoais\Doutorado\Cefet\data
📊 Arquivos encontrados: 7


1. 📄 bd_gta_dentro_mg202505091607.csv
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   📏 Dimensões:    8,298,490 linhas ×  15 colunas | 💾 Tamanho:     950.40 MB | ⚖️    0.0001 MB/linha | 🔑 Delimitador: ';'
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

   📊 ANÁLISE DETALHADA DE COLUNAS (15 total):
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

   [ 1] 📌 nr_gta
       ├─ Tipo: int64        (NUMÉRICO)
       ├─ Preenchimento: 100.00% (8,298,490/8,298,490)
       ├─ Únicos: 1000037 ( 12.05%)
       ├─ Intervalo: [1, 66760816]
       ├─ Centro: Média= 495154.2428 | Mediana= 494142.

C:\Users\leona\AppData\Local\Temp\ipykernel_15956\347599314.py:158: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho, delimiter=delimitador)



5. 📄 gtas_com_distancias.csv
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   📏 Dimensões:    8,363,167 linhas ×  21 colunas | 💾 Tamanho:    1156.59 MB | ⚖️    0.0001 MB/linha | 🔑 Delimitador: ';'
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

   📊 ANÁLISE DETALHADA DE COLUNAS (21 total):
   ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

   [ 1] 📌 nr_gta
       ├─ Tipo: int64        (NUMÉRICO)
       ├─ Preenchimento: 100.00% (8,363,167/8,363,167)
       ├─ Únicos: 1000067 ( 11.96%)
       ├─ Intervalo: [1, 66760816]
       ├─ Centro: Média= 495055.4588 | Mediana= 493920.0000
       ├─ Dispersão: σ=293024.4854 | σ²=85863349061.8536 | CV=  59.19%
       ├─ Quartis: Q1=241777.0000 | Q2=493920.0000 | Q3=746996.0000 | IQR=505219.0000
       ├─ Forma: Assimetria=  3.7872 | C

## Passo 3: Resumo de Melhorias e Finalização

In [3]:
print("\n" + "="*120)
print("📊 EXPLORADOR DE DADOS CSV - VERSÃO OTIMIZADA E PODEROSA")
print("="*120)
print("""
✨ MELHORIAS IMPLEMENTADAS NO EXPLORADOR:

1. ✅ FUNÇÃO UNIFICADA
   └─ Consolidou 4 funções redundantes em 1 única função: explorador_csv_completo()
   
2. ✅ SEM REDUNDÂNCIAS
   └─ Eliminou processamento duplicado e código repetido

3. ✅ 30+ MÉTRICAS POR COLUNA
   ├─ Informações sobre tipo de dados (Python type, numérico/categórico)
   ├─ Completude: valores totais, válidos, nulos, taxa de preenchimento
   ├─ Intervalo: mín, máx, amplitude, range textual
   ├─ Tendência Central: média, mediana, moda
   ├─ Dispersão: desvio padrão, variância, CV%
   ├─ Forma: Assimetria (skewness), Curtose (kurtosis)
   ├─ Quartis: Q1, Q2, Q3, IQR, limites de outliers (IQR)
   ├─ Percentis: 5º e 95º percentil
   ├─ Taxa de Unicidade
   └─ Para categóricos: Top 5, comprimento, diversidade

4. ✅ ANÁLISE COMPLETA DE TIPOS
   ├─ NUMÉRICO: Todas as 24+ métricas estatísticas
   └─ CATEGÓRICO: Frequência, comprimento, distribuição

5. ✅ FORMATAÇÃO VISUAL PROFISSIONAL
   ├─ Console: Saída com emojis, tabelas alinhadas
   ├─ Estrutura em árvore para legibilidade
   └─ Separadores visuais entre seções

6. ✅ MÚLTIPLOS FORMATOS DE SAÍDA
   ├─ Console: Exibição em tempo real com formatação
   ├─ CSV: 'analise_completa_colunas.csv' - Todas as métricas
   └─ TXT: 'RELATORIO_EXPLORACAO_COMPLETO.txt' - Relatório estruturado

════════════════════════════════════════════════════════════════════════════════

MODO DE USO:

    # Executar exploração completa
    dados = explorador_csv_completo(
        pasta=r'D:\\OneDrive\\Pessoais\\Doutorado\\Cefet\\data',
        salvar_relatorios=True,   # Salva em CSV
        gerar_txt=True             # Gera arquivo TXT
    )

ARQUIVOS GERADOS EM 'relatorio/resultados/informacoes_dados/':
  1. analise_completa_colunas.csv  → Tabela com todas as 30+ métricas
  2. RELATORIO_EXPLORACAO_COMPLETO.txt → Relatório estruturado

════════════════════════════════════════════════════════════════════════════════
""")
print("="*120)


📊 EXPLORADOR DE DADOS CSV - VERSÃO OTIMIZADA E PODEROSA

✨ MELHORIAS IMPLEMENTADAS NO EXPLORADOR:

1. ✅ FUNÇÃO UNIFICADA
   └─ Consolidou 4 funções redundantes em 1 única função: explorador_csv_completo()

2. ✅ SEM REDUNDÂNCIAS
   └─ Eliminou processamento duplicado e código repetido

3. ✅ 30+ MÉTRICAS POR COLUNA
   ├─ Informações sobre tipo de dados (Python type, numérico/categórico)
   ├─ Completude: valores totais, válidos, nulos, taxa de preenchimento
   ├─ Intervalo: mín, máx, amplitude, range textual
   ├─ Tendência Central: média, mediana, moda
   ├─ Dispersão: desvio padrão, variância, CV%
   ├─ Forma: Assimetria (skewness), Curtose (kurtosis)
   ├─ Quartis: Q1, Q2, Q3, IQR, limites de outliers (IQR)
   ├─ Percentis: 5º e 95º percentil
   ├─ Taxa de Unicidade
   └─ Para categóricos: Top 5, comprimento, diversidade

4. ✅ ANÁLISE COMPLETA DE TIPOS
   ├─ NUMÉRICO: Todas as 24+ métricas estatísticas
   └─ CATEGÓRICO: Frequência, comprimento, distribuição

5. ✅ FORMATAÇÃO VISUAL PR